# GeoGuesser-RAG: Retrieval-Augmented Generation Game
This notebook demonstrates a simple GeoGuesser-style game using Retrieval-Augmented Generation (RAG). The application retrieves hints about a country from a dataset and challenges the user to guess the correct country.

**Step 1:** Import all necessary libraries (pandas for data handling, sentence-transformers for generating embeddings, FAISS for similarity search).


In [27]:


import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from IPython.display import Image, display



**Step 2:** Load the country dataset from `test_countries.csv` and display the first few rows to verify the structure.


In [28]:
# Load the countries dataset
df = pd.read_csv("test/test_countries.csv")
df.head()


,Country,Capital,Continent,Population,Official_Language,Fun_Fact,Flag_URL
0,Switzerland,Bern,Europe,8500000,"German, French, Italian, Romansh",Four official languages,https://upload.wikimedia.org/wikipedia/commons...
1,Japan,Tokyo,Asia,126000000,Japanese,Has over 6000 islands,https://upload.wikimedia.org/wikipedia/en/9/9e...
2,Brazil,Brasilia,South America,210000000,Portuguese,Largest coffee producer,https://upload.wikimedia.org/wikipedia/en/0/05...
3,Canada,Ottawa,North America,38000000,English/French,Second largest country,https://upload.wikimedia.org/wikipedia/commons...
4,Egypt,Cairo,Africa,100000000,Arabic,Home of the ancient pyramids,https://upload.wikimedia.org/wikipedia/commons...


**Step 3:** Prepare the text that will be used to generate hints. We combine the most useful fields into one string per country.


In [29]:
# Combine relevant fields to create hint text for each country
df['hint_text'] = df.apply(lambda row: f"Capital: {row['Capital']}, Continent: {row['Continent']}, Population: {row['Population']}, Language: {row['Official_Language']}, Fun Fact: {row['Fun_Fact']}", axis=1)
df[['Country', 'hint_text']].head()


,Country,hint_text
0,Switzerland,"Capital: Bern, Continent: Europe, Population: ..."
1,Japan,"Capital: Tokyo, Continent: Asia, Population: 1..."
2,Brazil,"Capital: Brasilia, Continent: South America, P..."
3,Canada,"Capital: Ottawa, Continent: North America, Pop..."
4,Egypt,"Capital: Cairo, Continent: Africa, Population:..."


**Step 4:** Generate embeddings for each country hint using a pre-trained Sentence Transformer. These embeddings will be used for similarity search in the RAG pipeline.


In [30]:
# Load the sentence transformer model (can be changed later for better performance)
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

# Generate embeddings for each hint_text
embeddings = model.encode(df['hint_text'].tolist(), show_progress_bar=True)


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.37it/s]


**Step 5:** Create a FAISS vector index and add all country embeddings to it. This allows for fast similarity search based on user queries.


In [31]:
# Create a FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))


**Step 6:** Define a retrieval function. Given a user query (or a guess/hint), the function finds the most relevant countries based on the FAISS index.


In [32]:
def retrieve_country_hint(query, top_k=3):
    # Encode the user query
    query_embedding = model.encode([query])
    # Search for the most similar countries
    D, I = index.search(np.array(query_embedding), top_k)
    # Return top-k results
    return df.iloc[I[0]][['Country', 'hint_text']]

# Example query
retrieve_country_hint("This country has four official languages and is in Europe.", top_k=3)


,Country,hint_text
0,Switzerland,"Capital: Bern, Continent: Europe, Population: ..."
5,Australia,"Capital: Canberra, Continent: Oceania, Populat..."
7,Germany,"Capital: Berlin, Continent: Europe, Population..."


## Game Loop: Guess the Country
In this basic version, the notebook randomly selects a country and presents its hints. The user must type their guess, and the code checks if it is correct.


In [33]:
import random

def play_geoguesser_game():
    score = 0
    rounds = 5  # Number of rounds to play

    for i in range(rounds):
        print(f"\n--- Round {i+1} ---")
        # Randomly select a country
        idx = random.randint(0, len(df)-1)
        selected = df.iloc[idx]
        print("Hint:")
        print(selected['hint_text'])
        display(Image(url=selected['Flag_URL']))


        
        user_guess = input("Your guess: ").strip()
        
        if user_guess.lower() == selected['Country'].lower():
            print("✅ Correct!")
            score += 1
        else:
            print(f"❌ Wrong. The correct answer was: {selected['Country']}")
    
    print(f"\nGame over! Your score: {score} / {rounds}")

# Start the game
play_geoguesser_game()



--- Round 1 ---
Hint:
Capital: Brasilia, Continent: South America, Population: 210000000, Language: Portuguese, Fun Fact: Largest coffee producer


❌ Wrong. The correct answer was: Brazil

--- Round 2 ---
Hint:
Capital: Ottawa, Continent: North America, Population: 38000000, Language: English/French, Fun Fact: Second largest country


❌ Wrong. The correct answer was: Canada

--- Round 3 ---
Hint:
Capital: Ottawa, Continent: North America, Population: 38000000, Language: English/French, Fun Fact: Second largest country


❌ Wrong. The correct answer was: Canada

--- Round 4 ---
Hint:
Capital: Pretoria, Continent: Africa, Population: 59000000, Language: Zulu, Xhosa, Afrikaans, English, Fun Fact: Known for its diversity of cultures


❌ Wrong. The correct answer was: South Africa

--- Round 5 ---
Hint:
Capital: Bern, Continent: Europe, Population: 8500000, Language: German, French, Italian, Romansh, Fun Fact: Four official languages


❌ Wrong. The correct answer was: Switzerland

Game over! Your score: 0 / 5


**Game Logic:**
- The function randomly selects a country from the dataset for each round.
- The user is presented with a hint and types their guess.
- The answer is checked for correctness (case insensitive).
- The game tracks the score and displays it at the end.


In [34]:
import gradio as gr
import pandas as pd
import random

# Load your countries data
df = pd.read_csv("test_countries.csv")

def generate_new_question():
    idx = random.randint(0, len(df)-1)
    row = df.iloc[idx]
    # Prepare the hint text and flag URL
    hint = f"Capital: {row['Capital']}\nContinent: {row['Continent']}\nPopulation: {row['Population']}\nLanguage: {row['Official_Language']}\nFun Fact: {row['Fun_Fact']}"
    flag_url = row['Flag_URL']
    solution = row['Country']
    return hint, flag_url, solution

# Gradio state: [hint, flag_url, solution, score, round]
def start_game():
    hint, flag_url, solution = generate_new_question()
    return [hint, flag_url, solution, 0, 1], hint, flag_url, "", "Start guessing!"

def guess_country(user_guess, state):
    hint, flag_url, solution, score, round_ = state
    feedback = ""
    user_guess = user_guess.strip()
    if user_guess.lower() == solution.lower():
        score += 1
        feedback = f"✅ Correct! It was {solution}."
    else:
        feedback = f"❌ Wrong. The correct answer was {solution}."

    # Next round
    hint, flag_url, solution = generate_new_question()
    round_ += 1
    state = [hint, flag_url, solution, score, round_]
    return state, hint, flag_url, "", f"{feedback} Your score: {score} / {round_-1}"

with gr.Blocks() as demo:
    gr.Markdown("# 🌍 GeoGuesser-RAG\nGuess the country from hints and flag!")
    state = gr.State()
    with gr.Row():
        with gr.Column():
            hint_box = gr.Markdown()
            flag_img = gr.Image(type="pil", elem_id="flag_img", show_label=False)
        with gr.Column():
            input_box = gr.Textbox(label="Your guess", placeholder="Enter country name")
            feedback_box = gr.Markdown()

    # Button to submit guess
    guess_btn = gr.Button("Submit Guess")

    # Start game button
    start_btn = gr.Button("Start New Game")

    # Start game on click
    start_btn.click(fn=start_game, outputs=[state, hint_box, flag_img, input_box, feedback_box])
    # Submit guess
    guess_btn.click(fn=guess_country, inputs=[input_box, state], outputs=[state, hint_box, flag_img, input_box, feedback_box])

demo.launch()


* Running on local URL:  http://127.0.0.1:7863

To create a public link, set `share=True` in `launch()`.


In [35]:
import gradio as gr
import pandas as pd
import random

# Load your countries data
df = pd.read_csv("test_countries.csv")

def generate_new_question(game_mode):
    idx = random.randint(0, len(df)-1)
    row = df.iloc[idx]
    hint = f"Capital: {row['Capital']}\nContinent: {row['Continent']}\nPopulation: {row['Population']}\nLanguage: {row['Official_Language']}\nFun Fact: {row['Fun_Fact']}"
    flag_url = row['Flag_URL']
    solution = row['Country']
    if game_mode == "Guess by Hint":
        return hint, None, solution
    elif game_mode == "Guess by Flag":
        return None, flag_url, solution
    else:
        return hint, flag_url, solution

def start_game(game_mode):
    hint, flag_url, solution = generate_new_question(game_mode)
    flag_visible = True if flag_url else False
    hint_visible = True if hint else False
    return [hint, flag_url, solution, 0, 1, game_mode], \
           (hint if hint else ""), \
           gr.update(value=flag_url if flag_url else None, visible=flag_visible), \
           "", \
           f"Mode: {game_mode} — Start guessing!", \
           gr.update(visible=hint_visible), \
           gr.update(visible=flag_visible)

def guess_country(user_guess, state):
    hint, flag_url, solution, score, round_, game_mode = state
    feedback = ""
    user_guess = user_guess.strip()
    if user_guess.lower() == solution.lower():
        score += 1
        feedback = f"✅ Correct! It was {solution}."
    else:
        feedback = f"❌ Wrong. The correct answer was {solution}."

    # Next round
    new_hint, new_flag_url, new_solution = generate_new_question(game_mode)
    flag_visible = True if new_flag_url else False
    hint_visible = True if new_hint else False
    round_ += 1
    state = [new_hint, new_flag_url, new_solution, score, round_, game_mode]
    return state, \
           (new_hint if new_hint else ""), \
           gr.update(value=new_flag_url if new_flag_url else None, visible=flag_visible), \
           "", \
           f"{feedback} Your score: {score} / {round_-1}", \
           gr.update(visible=hint_visible), \
           gr.update(visible=flag_visible)

with gr.Blocks() as demo:
    gr.Markdown("# 🌍 GeoGuesser-RAG\nChoose your game mode!")
    state = gr.State()
    game_mode_selector = gr.Radio(choices=["Guess by Hint", "Guess by Flag"], label="Select Game Mode", value="Guess by Hint")
    with gr.Row():
        with gr.Column():
            hint_box = gr.Markdown(visible=True)
            flag_img = gr.Image(type="pil", elem_id="flag_img", show_label=False, visible=True)
        with gr.Column():
            input_box = gr.Textbox(label="Your guess", placeholder="Enter country name")
            feedback_box = gr.Markdown()
    guess_btn = gr.Button("Submit Guess")
    start_btn = gr.Button("Start New Game")

    # Start game when clicking the button
    start_btn.click(fn=start_game, inputs=[game_mode_selector], 
                    outputs=[state, hint_box, flag_img, input_box, feedback_box, hint_box, flag_img])
    # Start game immediately when mode changes
    game_mode_selector.change(fn=start_game, inputs=[game_mode_selector],
                    outputs=[state, hint_box, flag_img, input_box, feedback_box, hint_box, flag_img])
    # Submit guess
    guess_btn.click(fn=guess_country, inputs=[input_box, state], 
                    outputs=[state, hint_box, flag_img, input_box, feedback_box, hint_box, flag_img])

demo.launch()


* Running on local URL:  http://127.0.0.1:7864

To create a public link, set `share=True` in `launch()`.
